<a href="https://colab.research.google.com/github/yzhai64/fh-agents-course/blob/main/%E2%80%9Cmultiagent_notebook_ipynb%E2%80%9D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solving a complex task with a multi-agent hierarchy

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

The reception is approaching! With your help, Alfred is now nearly finished with the preparations.

But now there's a problem: the Batmobile has disappeared. Alfred needs to find a replacement, and find it quickly.

Fortunately, a few biopics have been done on Bruce Wayne's life, so maybe Alfred could get a car left behind on one of the movie set, and re-engineer it up to modern standards, which certainly would include a full self-driving option.

But this could be anywhere in the filming locations around the world - which could be numerous.

So Alfred wants your help. Could you build an agent able to solve this task?

> 👉 Find all Batman filming locations in the world, calculate the time to transfer via boat to there, and represent them on a map, with a color varying by boat transfer time. Also represent some supercar factories with the same boat transfer time.

Let's build this!

In [48]:
!pip install 'smolagents[litellm]' matplotlib geopandas shapely kaleido -q

In [57]:
from huggingface_hub import notebook_login

notebook_login()

In [58]:
# We first make a tool to get the cargo plane transfer time.
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

22.82


For the model provider, we use Together AI, one of the new [inference providers on the Hub](https://huggingface.co/blog/inference-providers)!

Regarding the GoogleSearchTool: this requires either having setup env variable `SERPAPI_API_KEY` and passing `provider="serpapi"` or having `SERPER_API_KEY` and passing `provider=serper`.

If you don't have any Serp API provider setup, you can use `DuckDuckGoSearchTool` but beware that it has a rate limit.

In [59]:
import os
from PIL import Image
from smolagents import CodeAgent, GoogleSearchTool, HfApiModel, VisitWebpageTool


model = HfApiModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

We can start with creating a baseline, simple agent to give us a simple report.

In [60]:
task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time."""

In [61]:
from google.colab import userdata
import os
os.environ["SERPAPI_API_KEY"] = userdata.get('SERPAPI_API_KEY')

In [62]:
agent = CodeAgent(
    model=model,
    tools=[GoogleSearchTool(), VisitWebpageTool(), calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps=20,
)

In [63]:
result = agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
402 Client Error: Payment Required for url: https://huggingface.co/api/inference-proxy/together/v1/chat/completions
(Request ID: Root=1-67cfccb0-047ea5c760b72b7d65f22c1b;78ddd30c-2e35-4bed-805a-1b83bdf18ebd)

You have exceeded your monthly included credits for Inference Providers. Pay-as-you-go above your included PRO 
quota will be available soon.

[Step 1: Duration 0.09 seconds]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 20 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
402 Client Error: Payment Required for url: https://huggingface.co/api/inference-proxy/together/v1/chat/completions
(Request ID: Root=1-67cfccb2-151a241775f8a710489d5bc9;131c99dd-8792-439d-813f-9f55b278f0b5)

You have exceeded your monthly included credits for Inference Providers. Pay-as-you-go above your included PRO 
quota will be available soon.

[Step 20: Duration 0.12 seconds]

Reached max steps.

[Step 21: Duration 0.21 seconds]

In [ ]:
result

,Location,Travel Time to Gotham (hours)
0,"Necropolis Cemetery, Glasgow, Scotland, UK",8.60
1,"St. George's Hall, Liverpool, England, UK",8.81
2,"Two Temple Place, London, England, UK",9.17
3,"Wollaton Hall, Nottingham, England, UK",9.00
4,"Knebworth House, Knebworth, Hertfordshire, Eng...",9.15
5,"Acton Lane Power Station, Acton Lane, Acton, E...",9.16
6,"Queensboro Bridge, New York City, USA",1.01
7,"Wall Street, New York City, USA",1.00
8,"Mehrangarh Fort, Jodhpur, Rajasthan, India",18.34
9,"Turda Gorge, Turda, Romania",11.89


We could already improve this a bit by throwing in some dedicated planning steps, and adding more prompting.

In [ ]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're an expert analyst. You make comprehensive reports after visiting many websites.                          │
│ Don't hesitate to search for many queries at once in a for loop.                                                │
│ For each data point that you find, visit the source url to confirm numbers.                                     │
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here is the plan of action that I will follow to solve the task:
```
1. Perform a web search to find a comprehensive list of Batman filming locations worldwide.
2. For each filming location, visit the source URL to confirm the coordinates.
3. Perform a web search to find a comprehensive list of supercar factories worldwide.
4. For each supercar factory, visit the source URL to confirm the coordinates.
5. Calculate the cargo plane transfer time from Gotham to each Batman filming location.
6. Calculate the cargo plane transfer time from Gotham to each supercar factory.
7. Organize the collected data into a pandas dataframe with columns for location name, coordinates, distance from 
Gotham, and estimated transfer time.
8. Finalize the pandas dataframe to include both Batman filming locations and supercar factories with the same 
cargo plane transfer time.
9. Provide the final answer as the pandas dataframe.


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  batman_filming_locations = web_search(query="Batman filming locations worldwide")                                
  print(batman_filming_locations)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results
0. [12 Batman Movie Locations You Can 
Visit!](https://www.travelandleisureasia.com/global/destinations/batman-movie-locations-you-can-visit/)
Date published: Jan 10, 2023
Source: Travel and Leisure Asia

Most of the filming of Batman movies is done in the Warner Bros studios and across the US, including New York and 
Pittsburgh. You will notice ...

1. [Category:Film Locations | Batman Wiki - Fandom](https://batman.fandom.com/wiki/Category:Film_Locations)
Source: Batman Wiki

Film Locations · A. Arctic World · B Batcave (Burtonverse) · C Cave of Horrors · D Category:Dark Knight Trilogy 
locations · F.

2. [The Batman (2022) - Filming & production](https://www.imdb.com/title/tt1877830/locations/)
Source: IMDb

Filming locations: Necropolis Cemetery, Glasgow, Scotland, UK (Batman and Selina leaving the cemetery) Helpful•86 1
St. George's Hall, Liverpool, England, UK

3. [The Batman | Film Locations](https://movie-locations.com/movies/b/The-Batman-2022-2.php)
Source: The Worldwide Guide To Movie Locations

Film locations for The Batman (2022) in Liverpool, London, Glasgow and Chicago.

4. [Dark Knight Rises Tour: See Batman Movie 
Locations](https://www.travelchannel.com/interests/arts-and-culture/photos/see-batman-movie-locations)
Source: Travel Channel

See Batman Movie Locations · Wollaton Hall · Carnegie Mellon University · The Farmiloe Building · Queensboro Bridge
· Wall Street · Mehrangarh Fort · Turda Saline.

5. [What cities in America and anywhere in the world do you 
...](https://www.reddit.com/r/batman/comments/1d1t88q/what_cities_in_america_and_anywhere_in_the_world/)
Source: Reddit · r/batman

Glasgow and Liverpool were used in the shoot for The Batman. I'd avoid Chicago, as that looks more like Metropolis 
than Gotham.

6. [Where was The Batman filmed? ALL the Filming Locations 
...](https://www.atlasofwonders.com/2022/04/where-was-the-batman-filmed.html)
Source: Atlas of Wonders

The Batman was primarily filmed in the United Kingdom. Most of the recognizable buildings of this new version of 
Gotham City are located in Liverpool.

7. [The Dark Knight | 2008](https://movie-locations.com/movies/d/Dark-Knight.php)
Source: The Worldwide Guide To Movie Locations

Discover where The Dark Knight (2008) was filmed around Chicago, as well as in London and Bedfordshire in the UK, 
and briefly in Hong Kong.

8. [The Dark Knight Rises](https://onlocationtours.com/locations/the-dark-knight-rises/)
Source: On Location Tours

The 2012 end of Christopher Nolan's Batman trilogy, The Dark Knight Rises, was filmed in New York City. Filming 
locations can be seen on The Super Tour of NYC.

9. [The Worldwide Guide to Movie Locations - Batman Begins 
...](https://m.facebook.com/MovieLocations/photos/batman-begins-2005-senate-house-university-of-london-bloomsbury-l
ondon-wc1-bruce/940979554684554/)
Source: Facebook · The Worldwide Guide to Movie Locations

The Worldwide Guide to Movie Locations - Batman Begins (2005) Senate House, University of London, Bloomsbury, 
London WC1. Bruce Wayne ...

Out: None

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Batman filming locations                                                                                       
  batman_locations = [                                                                                             
      {"name": "St George's Hall, Liverpool, UK", "coordinates": None},                                            
      {"name": "Necropolis Cemetery, Glasgow, UK", "coordinates": None},                                           
      {"name": "Anfield Cemetery, Liverpool, UK", "coordinates": None},                                            
      {"name": "One Liberty Plaza, New York, USA", "coordinates": None},                                           
      {"name": "Trump Tower, New York, USA", "coordinates": None}                                                  
  ]                                                                                                                
                                                                                                                   
  # Supercar factories                                                                                             
  supercar_factories = [                                                                                           
      {"name": "Bugatti - Molsheim, France", "coordinates": None},                                                 
      {"name": "Ferrari - Maranello, Italy", "coordinates": None},                                                 
      {"name": "Lamborghini - Sant'Agata Bolognese, Italy", "coordinates": None},                                  
      {"name": "Pagani - San Cesario sul Panaro, Italy", "coordinates": None},                                     
      {"name": "McLaren - Woking, UK", "coordinates": None},                                                       
      {"name": "Porsche - Stuttgart, Germany", "coordinates": None},                                               
      {"name": "Aston Martin - Gaydon, UK", "coordinates": None},                                                  
      {"name": "Koenigsegg - Angered, Sweden", "coordinates": None},                                               
      {"name": "Rimac - Zagreb, Croatia", "coordinates": None},                                                    
      {"name": "W Motors - Dubai, UAE", "coordinates": None}                                                       
  ]                                                                                                                
                                                                                                                   
  # Perform web searches to find coordinates                                                                       
  for location in batman_locations + supercar_factories:                                                           
      search_query = f"[38;2;230;219;116;48;2;39;

Execution logs:
Batman Locations: [{'name': "St George's Hall, Liverpool, UK", 'coordinates': "## Search Results\n0. [St George's 
Hall, Liverpool](https://en.wikipedia.org/wiki/St_George%27s_Hall,_Liverpool)\nSource: Wikipedia\n\nSt George's 
Place, Liverpool, England · 53°24′31″N 2°58′48″W\ufeff / \ufeff53.4086°N 2.9801°W\ufeff / 53.4086; -2.9801 · SJ 349
907 · 1841–1854.\n\n1. [Where is Liverpool, the UK on Map Lat Long 
Coordinates](https://www.latlong.net/place/liverpool-the-uk-28017.html)\nSource: Latitude and Longitude 
Finder\n\nThe latitude of Liverpool, the UK is 53.400002, and the longitude is -2.983333. Liverpool, the UK is 
located at United Kingdom country in the Cities place ...\n\n2. [St George's Hall, Non Civil Parish - 
1361677](https://historicengland.org.uk/listing/the-list/list-entry/1361677)\nSource: Historic England\n\nSt 
George's Hall ; Date first listed: 28-Jun-1952 ; List Entry Name: St George's Hall ; Statutory Address: St George's
Hall, St George's Place, Liverpool, L1 1JJ.\n\n3. [St Georges 
Hall](https://www.municipalhotelliverpool.com/places/st-georges-hall/)\nSource: The Municipal Hotel & Spa 
Liverpool\n\nAddress St Georges Place L1 1JJ Liverpool United Kingdom GPS 53.4084932, -2.9824465. Hotel. Point of 
interest. Find a route. From Insert starting address.\n\n4. [St George's 
Hall](https://www.atlasobscura.com/places/st-georges-hall)\nDate published: Sep 21, 2018\nSource: Atlas 
Obscura\n\nSt George's Hall was built in Liverpool between 1841 and 1851. ... Liverpool, England United Kingdom. 
Copy Address. 53.408419, -2.980235.\n\n5. [Travelling to St George's Hall | Liverpool John Moores 
...](https://www.ljmu.ac.uk/study/visit-us/directions/st-georges-hall)\nSource: Liverpool John Moores 
University\n\nGet directions for travelling to St George's Hall, including sat nav coordinates, parking information
and public transport information.\n\n6. [Liverpool 
Cenotaph](https://en.wikipedia.org/wiki/Liverpool_Cenotaph)\nSource: Wikipedia\n\nSt George's Plateau, Liverpool, 
England · 53°24′31″N 2°58′46″W\ufeff / \ufeff53.4085°N 2.9795°W\ufeff / 53.4085; -2.9795 · 1927–30 · Lionel Bailey 
Budden.\n\n7. [Discover - Welcome to St George's Hall, 
Liverpool](https://stgeorgeshallliverpool.co.uk/discover/)\nSource: St George's Hall\n\nSituated opposite Lime 
Street station, St George's Hall forms an intrinsic part of Liverpool's William Brown Conservation Area and 
provides a magnificent ...\n\n8. [Map of Liverpool, United Kingdom showing 
...](https://latitude.to/map/gb/united-kingdom/cities/liverpool/articles/page/14)\nSource: Latitude.to\n\nMap of 
Liverpool, United Kingdom showing latitude and longitude of items of interest. Page 14 of 47.\n\n9. [Contact & 
Accessibility Information](https://stgeorgeshallliverpool.co.uk/contact-and-accessibility-information/)\nSource: St
George's Hall\n\nThe Visitor Entrance (accessible via St John's Lane, L1 1HF) is open Monday to Saturday from 
9.30am to 5pm (last entry 4.45pm)."}, {'name': 'Necropolis Cemetery, Glasgow, UK', 'coordinates': '## Search 
Results\n0. [Glasgow Necropolis](https://en.wikipedia.org/wiki/Glasgow_Necropolis)\nSource: Wikipedia\n\nThe 
Glasgow Necropolis is a Victorian cemetery in Glasgow, Scotland. It is ... Coordinates · 55°51′44″N 4°14′00″W\ufeff
/ \ufeff55.86217°N 4.23340°W\ufeff / 55.86217 ...\n\n1. [GPS coordinates of Glasgow Necropolis, United Kingdom. 
...](https://latitude.to/articles-by-country/gb/united-kingdom/18217/glasgow-necropolis)\nSource: 
Latitude.to\n\nThe Glasgow Necropolis is a Victorian cemetery in Glasgow, Scotland. It is on a low but very 
prominent hill to the east of Glasgow Cathedral.\n\n2. [Glasgow Necropolis | Cemetery 
Details](https://www.cwgc.org/visit-us/find-cemeteries-memorials/cemetery-details/2081112/glasgow-necropolis/)\nSou
rce: The Commonwealth War Graves Commission | CWGC\n\nGlasgow; Identified casualties from First & Second World War;
GPS Coordinates Latitude: 55.86217, Longitude: -4.23335. GLASGOW 

[Step 4: Duration 60.24 seconds| Input tokens: 33,851 | Output tokens: 2,693]

[Step 5: Duration 0.68 seconds| Input tokens: 55,750 | Output tokens: 3,824]

HfHubHTTPError: 422 Client Error: Unprocessable Entity for url: https://huggingface.co/api/inference-proxy/together/v1/chat/completions (Request ID: Root=1-67cfbd5e-62c3de10773325aa36db4c31;eff7cd40-0435-491c-bb43-82a0d3a10e6e)

{'message': 'Input validation error: `inputs` tokens + `max_new_tokens` must be <= 32769. Given: 35975 `inputs` tokens and 8096 `max_new_tokens`', 'type': 'invalid_request_error', 'param': None, 'code': None}
Make sure 'conversational' task is supported by the model.

In [ ]:
detailed_report

,Location,Travel Time (hours)
0,"Bridge of Sighs, Glasgow Necropolis, Glasgow, ...",8.6
1,"Wishart Street, Glasgow, Scotland, UK",8.6


Thanks to these quick changes, we obtained a much more concise report by simply providing our agent a detailed prompt, and giving it planning capabilities!

💸 But as you can see, the context window is quickly filling up. So **if we ask our agent to combine the results of detailed search with another, it will be slower and quickly ramp up tokens and costs**.

➡️ We need to improve the structure of our system.

## ✌️ Splitting the task between two agents

Multi-agent structures allow to separate memories between different sub-tasks, with two great benefits:
- Each agent is more focused on its core task, thus more performant
- Separating memories reduces the count of input tokens at each step, thus reducing latency and cost.

Let's create a team with a dedicated web search agent, managed by another agent.

The manager agent should have plotting capabilities to redact its final report: so let us give it access to additional imports, including `matplotlib`, and `geopandas` + `shapely` for spatial plotting.

In [ ]:
import os
os.environ["SERPER_API_KEY"] = "89cbcd5b0719acd335594cbd0f86eec069204d6e9878aa989825b27ed3466245"
model = HfApiModel(
    "Qwen/Qwen2.5-Coder-32B-Instruct", provider="together", max_tokens=8096
)

web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serper"),
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

The manager agent will need to do some mental heavy lifting.

So we give it the stronger model [DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1), and add a `planning_interval` to the mix.

In [ ]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
from smolagents.utils import encode_image_base64, make_image_url
from smolagents import OpenAIServerModel


def check_reasoning_and_plot(final_answer, agent_memory):
    final_answer
    multimodal_model = OpenAIServerModel("gpt-4o", max_tokens=8096)
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
    image = Image.open(filepath)
    prompt = (
        f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
        "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
        "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
        "Don't be harsh: if the plot mostly solves the task, it should pass."
        "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
                {
                    "type": "image_url",
                    "image_url": {"url": make_image_url(encode_image_base64(image))},
                },
            ],
        }
    ]
    output = multimodal_model(messages).content
    print("Feedback: ", output)
    if "FAIL" in output:
        raise Exception(output)
    return True


manager_agent = CodeAgent(
    model=HfApiModel("deepseek-ai/DeepSeek-R1", provider="together", max_tokens=8096),
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "shapely",
        "json",
        "pandas",
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_reasoning_and_plot],
    max_steps=15,
)

Let us inspect what this team looks like:

In [ ]:
manager_agent.visualize()

CodeAgent | deepseek-ai/DeepSeek-R1
├── ✅ Authorized imports: ['geopandas', 'plotly', 'shapely', 'json', 'pandas', 'numpy']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name                        ┃ Description                           ┃ Arguments                             ┃
│   ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ calculate_cargo_travel_time │ Calculate the travel time for a cargo │ origin_coords (`array`): Tuple of     │
│   │                             │ plane between two points on Earth     │ (latitude, longitude) for the         │
│   │                             │ using great-circle distance.          │ starting point                        │
│   │                             │                                       │ destination_coords (`array`): Tuple   │
│   │                             │                                       │ of (latitude, longitude) for the      │
│   │                             │                                       │ destination                           │
│   │                             │                                       │ cruising_speed_kmh (`number`):        │
│   │                             │                                       │ Optional cruising speed in km/h       │
│   │                             │                                       │ (defaults to 750 km/h for typical     │
│   │                             │                                       │ cargo planes)                         │
│   │ final_answer                │ Provides a final answer to the given  │ answer (`any`): The final answer to   │
│   │                             │ problem.                              │ the problem                           │
│   └─────────────────────────────┴───────────────────────────────────────┴───────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | Qwen/Qwen2.5-Coder-32B-Instruct
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Browses the web to find information
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                        ┃ Description                       ┃ Arguments                         ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search                  │ Performs a google web search for  │ query (`string`): The search      │
            │                             │ your query then returns a string  │ query to perform.                 │
            │                             │ of the top search results.        │ filter_year (`integer`):          │
            │                             │                                   │ Optionally restrict results to a  │
            │                             │                                   │ certain year                      │
            │ visit_webpage               │ Visits a webpage at the given url │ url (`string`): The url of the    │
            │                             │ and reads its content as a        │ webpage to visit.                 │
            │                             │ markdown string. Use this to      │                                   │
            │                             │ browse webpages.                  │                                   │
            │ calculate_cargo_travel_time │ Calculate the travel time for a   │ origin_coords (`array`): Tuple of │
            │                             │ cargo plane between two points on │ (latitude, longitude) for the     │
            │                             │ Earth using great-circle          │ starting point                    │
            │                             │ distance.     

In [ ]:
manager_agent.run("""
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_image.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W).                                                                             │
│ Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in     │
│ total.                                                                                                          │
│ Represent this as spatial map of the world, with the locations represented as scatter points with a color that  │
│ depends on the travel time, and save it to saved_map.png!                                                       │
│                                                                                                                 │
│ Here's an example of how to plot and return a map:                                                              │
│ import plotly.express as px                                                                                     │
│ df = px.data.carshare()                                                                                         │
│ fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,      │
│      color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)                                    │
│ fig.show()                                                                                                      │
│ fig.write_image("saved_image.png")                                                                              │
│ final_answer(fig)                                                                                               │
│                                                                                                                 │
│ Never try to process strings using code: when you have a string to read, just print it and you'll see it.       │
│                                                                                                                 │
╰─ HfApiModel - deepseek-ai/DeepSeek-R1 ──────────────────────────────────────────────────────────────────────────╯

[Step 1: Duration 0.05 seconds]

HfHubHTTPError: 402 Client Error: Payment Required for url: https://huggingface.co/api/inference-proxy/together/v1/chat/completions (Request ID: Root=1-67cfc1a4-4a93900b71bd340d29dbba51;f810e553-1531-4d2e-b8ad-b02fab0e5625)

You have exceeded your monthly included credits for Inference Providers. Pay-as-you-go above your included PRO quota will be available soon.

I don't know how that went in your run, but in mine, the manager agent skilfully divided tasks given to the web agent in `1. Search for Batman filming locations`, then `2. Find supercar factories`, before aggregating the lists and plotting the map.

Let's see what the map looks like by inspecting it directly from the agent state:

In [ ]:
manager_agent.python_executor.state["fig"]

![output map](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/smolagents/output_map.png)